In [3]:

import pandas as pd

# Load the CSV file from the current folder
df = pd.read_csv("retail_sales_dataset.csv")

# Display the results
print("Dataset loaded successfully!")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

display(df.head())

Dataset loaded successfully!
Number of rows: 1000
Number of columns: 9


,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100


In [4]:

df.head()

print("Column names:")

for column in df.columns:
    print(column)

df.info()

df.isnull().sum()


print("Duplicate rows:", df.duplicated().sum())

print(
    "Duplicate transaction IDs:",
    df["Transaction ID"].duplicated().sum()
)



Column names:
Transaction ID
Date
Customer ID
Gender
Age
Product Category
Quantity
Price per Unit
Total Amount
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    1000 non-null   int64 
 1   Date              1000 non-null   object
 2   Customer ID       1000 non-null   object
 3   Gender            1000 non-null   object
 4   Age               1000 non-null   int64 
 5   Product Category  1000 non-null   object
 6   Quantity          1000 non-null   int64 
 7   Price per Unit    1000 non-null   int64 
 8   Total Amount      1000 non-null   int64 
dtypes: int64(5), object(4)
memory usage: 70.4+ KB
Duplicate rows: 0
Duplicate transaction IDs: 0


In [5]:

# Convert the Date column to a proper datetime format
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Check the result
print("Date column data type:", df["Date"].dtype)
display(df[["Date"]].head())

invalid_dates = df["Date"].isna().sum()

print("Invalid or missing dates:", invalid_dates)

print("Invalid ages:", ((df["Age"] < 0) | (df["Age"] > 120)).sum())
print("Invalid quantities:", (df["Quantity"] <= 0).sum())
print("Invalid prices:", (df["Price per Unit"] < 0).sum())
print("Invalid total amounts:", (df["Total Amount"] < 0).sum())


df["Calculated Amount"] = (
    df["Quantity"] * df["Price per Unit"]
)

df["Amount Mismatch"] = (
    df["Total Amount"] != df["Calculated Amount"]
)

print(
    "Transactions with amount mismatches:",
    df["Amount Mismatch"].sum()
)

display(
    df.loc[
        df["Amount Mismatch"],
        [
            "Transaction ID",
            "Quantity",
            "Price per Unit",
            "Total Amount",
            "Calculated Amount"
        ]
    ].head()
)




Date column data type: datetime64[ns]


,Date
0,2023-11-24
1,2023-02-27
2,2023-01-13
3,2023-05-21
4,2023-05-06


Invalid or missing dates: 0
Invalid ages: 0
Invalid quantities: 0
Invalid prices: 0
Invalid total amounts: 0
Transactions with amount mismatches: 0


,Transaction ID,Quantity,Price per Unit,Total Amount,Calculated Amount


In [6]:

# Calculate important sales metrics

total_revenue = df["Total Amount"].sum()
total_transactions = df["Transaction ID"].nunique()
total_units_sold = df["Quantity"].sum()
average_transaction_value = df["Total Amount"].mean()

print("SALES SUMMARY")
print("-" * 35)
print("Total Revenue:", total_revenue)
print("Total Transactions:", total_transactions)
print("Total Units Sold:", total_units_sold)
print("Average Transaction Value:", round(average_transaction_value, 2))


SALES SUMMARY
-----------------------------------
Total Revenue: 456000
Total Transactions: 1000
Total Units Sold: 2514
Average Transaction Value: 456.0


In [7]:

category_summary = (
    df.groupby("Product Category")["Total Amount"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

category_summary.columns = ["Product Category", "Total Revenue"]

display(category_summary)


,Product Category,Total Revenue
0,Electronics,156905
1,Clothing,155580
2,Beauty,143515


In [8]:

df["Month"] = df["Date"].dt.to_period("M").astype(str)

monthly_summary = (
    df.groupby("Month")["Total Amount"]
    .sum()
    .reset_index()
)

monthly_summary.columns = ["Month", "Total Revenue"]

display(monthly_summary.head(12))


,Month,Total Revenue
0,2023-01,35450
1,2023-02,44060
2,2023-03,28990
3,2023-04,33870
4,2023-05,53150
5,2023-06,36715
6,2023-07,35465
7,2023-08,36960
8,2023-09,23620
9,2023-10,46580


In [9]:

gender_summary = (
    df.groupby("Gender")["Total Amount"]
    .agg(["sum", "count", "mean"])
    .reset_index()
)

gender_summary.columns = [
    "Gender",
    "Total Revenue",
    "Transactions",
    "Average Transaction Value"
]

display(gender_summary)


,Gender,Total Revenue,Transactions,Average Transaction Value
0,Female,232840,510,456.549020
1,Male,223160,490,455.428571


In [10]:

from pathlib import Path

OUTPUT_DIR = Path.cwd() / "output_reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output folder is ready:", OUTPUT_DIR)


Output folder is ready: C:\Users\ACER\Music\output_reports


In [11]:

import pandas as pd

# Create a summary table of key business metrics
sales_summary = pd.DataFrame({
    "Metric": [
        "Total Revenue",
        "Total Transactions",
        "Total Units Sold",
        "Average Transaction Value"
    ],
    "Value": [
        total_revenue,
        total_transactions,
        total_units_sold,
        round(average_transaction_value, 2)
    ]
})

# Create the Excel report
report_path = OUTPUT_DIR / "Automated_Sales_Report.xlsx"

with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
    sales_summary.to_excel(
        writer, sheet_name="Sales Summary", index=False
    )
    category_summary.to_excel(
        writer, sheet_name="Category Analysis", index=False
    )
    monthly_summary.to_excel(
        writer, sheet_name="Monthly Analysis", index=False
    )
    gender_summary.to_excel(
        writer, sheet_name="Gender Analysis", index=False
    )

print("Excel report created successfully!")
print("Saved at:", report_path)

Excel report created successfully!
Saved at: C:\Users\ACER\Music\output_reports\Automated_Sales_Report.xlsx


In [12]:

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

report_path = OUTPUT_DIR / "Automated_Sales_Report.xlsx"

wb = load_workbook(report_path)

for ws in wb.worksheets:
    # Format the header row
    for cell in ws[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill(
            fill_type="solid",
            fgColor="1F4E78"
        )
        cell.alignment = Alignment(
            horizontal="center"
        )

    # Adjust column widths automatically
    for column_cells in ws.columns:
        max_length = 0
        column_letter = column_cells[0].column_letter

        for cell in column_cells:
            if cell.value is not None:
                max_length = max(
                    max_length,
                    len(str(cell.value))
                )

        ws.column_dimensions[column_letter].width = min(
            max_length + 4, 35
        )

    # Add borders to cells
    thin_border = Border(
        bottom=Side(style="thin", color="D9E2F3")
    )

    for row in ws.iter_rows():
        for cell in row:
            cell.border = thin_border

    # Keep headers visible while scrolling
    ws.freeze_panes = "A2"

wb.save(report_path)

print("Excel report formatted successfully!")
print("Saved at:", report_path)


Excel report formatted successfully!
Saved at: C:\Users\ACER\Music\output_reports\Automated_Sales_Report.xlsx
